# Frozen model eICU evaluation and domain-shift/calibration descriptive analysis

This notebook contains code selected from the authors’ research notebooks because its analysis is described in the accepted COST manuscript. Outputs and execution counts have been removed. It may require authorized datasets, saved model artifacts, and variables established by preceding pipeline notebooks. See `docs/EXECUTION_ORDER.md`.


# Frozen model eICU evaluation and domain-shift/calibration descriptive analysis

This notebook documents one experiment reported in the COST manuscript.


In [ ]:
# CELL 1 — SETUP, PATHS, FLAGS
import os, sys, json, math, time, random, warnings, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Not running in Colab (or Drive already mounted):", e)

BASE       = Path(os.environ.get("COST_BASE", "/content/drive/MyDrive/COST_PROJECT"))
MIMIC_BASE = BASE / "mimic-iv-2.1"
HOSP       = MIMIC_BASE / "hosp"
ICU        = MIMIC_BASE / "icu"
CACHE_FILE = MIMIC_BASE / "mimic_cache" / "hf_cohort_processed.npz"
PRED_DIR   = MIMIC_BASE / "saved_predictions"
SAVE_DIR   = MIMIC_BASE / "saved_models"
REV_DIR    = MIMIC_BASE / "revision2_outputs"
CKPT_MAIN  = SAVE_DIR / "dsrnn_v2_best.pt"
ID_FILE    = REV_DIR / "REV6_FINAL_exact_identifiers.csv"
EICU_PRED  = REV_DIR / "external_eicu" / "eicu_external_predictions.npz"
OUT        = REV_DIR / "final_round_reviewer"
OUT.mkdir(parents=True, exist_ok=True)

# ---------------- FLAGS ----------------
RUN_COHORT_AUDIT         = True   # A1/A2: reads admissions.csv + icustays.csv (small, read-only)
RUN_CHECKPOINT_INFERENCE = True   # A4-A6: rebuild preprocessing + inference with frozen checkpoints
RUN_GATING_SEED_RECAL    = True   # A5: also recalibrate gating ON/OFF seed checkpoints (if *_best.pt exist)
RUN_EICU                 = True   # A8: only runs if EICU_PRED exists
RUN_OPTIONAL_TRAINING    = False  # A9: NEW TRAINING (≈1–2 h GPU). Keep False unless you decide to add it.
N_BOOT   = 2000
SEED     = 42
TRAIN_YEARS, VAL_YEARS, TEST_YEARS = range(2008, 2016), range(2016, 2018), range(2018, 2020)

VITAL_SIGNS = ['heart_rate', 'sbp', 'dbp', 'map', 'resp_rate', 'spo2', 'temperature']
LAB_VALUES  = ['creatinine', 'bnp', 'lactate', 'troponin', 'sodium', 'potassium', 'chloride',
               'bicarbonate', 'bun', 'glucose', 'hemoglobin', 'wbc', 'platelets', 'inr', 'pt',
               'ptt', 'ph', 'pao2', 'paco2', 'fio2', 'albumin', 'bilirubin_total', 'alt', 'ast']
STATIC_FEATURES = ['age', 'gender_male', 'admission_emergency', 'admission_urgent', 'elixhauser_score',
                   'hypertension', 'diabetes', 'ckd', 'copd', 'af', 'prior_hf']
ALL_TEMPORAL_FEATURES = VITAL_SIGNS + LAB_VALUES
N_FEATURES, N_STATIC = len(ALL_TEMPORAL_FEATURES), len(STATIC_FEATURES)
N_INPUT = N_FEATURES * 3

REPORT = {}      # key numbers for the response letter
FLAGS  = []      # audit notes to verify against the final pipeline

# Saved test predictions to evaluate: display name -> (file in PRED_DIR, key). Edit if your final files differ.
PRED_FILES = {
    "Logistic Regression": ("baseline_lr_predictions.npz", "probs"),
    "Random Forest":       ("baseline_rf_predictions.npz", "probs"),
    "Standard LSTM":       ("baseline_lstm_predictions.npz", "probs"),
    "Attention LSTM":      ("baseline_attn_lstm_predictions.npz", "probs"),
    "TCN":                 ("baseline_tcn_predictions.npz", "probs"),
    "COST (Initial)":      ("cost_before_predictions.npz", "probs"),
    "COST (Improved) single":   ("cost_improved_predictions.npz", "probs_single"),
    "COST (Improved) ensemble": ("cost_improved_predictions.npz", "probs_ensemble"),
}

GATING_SEEDS = [42, 43, 44]   # Revision II gating ON/OFF artefacts in REV_DIR (skipped if absent)

random.seed(SEED); np.random.seed(SEED)
for p in [CACHE_FILE, PRED_DIR, ID_FILE, CKPT_MAIN]:
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")
print("Output folder:", OUT)

In [ ]:
# CELL 2 — UTILITIES: metrics, DeLong, (cluster) bootstrap, calibration, recalibration, plots
def show(df, n=60):
    try:
        display(df.head(n))          # noqa: F821  (Colab/Jupyter)
    except NameError:
        print(df.head(n).to_string())

def save_table(df, name):
    df.to_csv(OUT / f"{name}.csv", index=False)
    TABLES[name] = df
    return df
TABLES = {}

def sens_at_spec(y, p, target_spec=0.90):
    # identical definition to the manuscript code
    fpr, tpr, _ = roc_curve(y, p)
    idx = np.argmin(np.abs((1 - fpr) - target_spec))
    return float(tpr[idx])

def _clip(p): return np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
def logit(p): p = _clip(p); return np.log(p / (1 - p))
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def fit_logistic(X, y, offset=None, iters=100):
    # Newton-Raphson logistic regression; X must include an intercept column if wanted
    X = np.asarray(X, float); y = np.asarray(y, float)
    off = np.zeros(len(y)) if offset is None else np.asarray(offset, float)
    b = np.zeros(X.shape[1])
    for _ in range(iters):
        mu = sigmoid(X @ b + off)
        W = mu * (1 - mu)
        H = X.T @ (X * W[:, None]) + 1e-9 * np.eye(X.shape[1])
        step = np.linalg.solve(H, X.T @ (y - mu))
        b += step
        if np.max(np.abs(step)) < 1e-8:
            break
    return b

def ece_score(y, p, n_bins=10):
    y = np.asarray(y, float); p = np.asarray(p, float)
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)

def calib_stats(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float); z = logit(p)
    prev = y.mean()
    slope = fit_logistic(np.c_[np.ones_like(z), z], y)[1]
    citl  = fit_logistic(np.ones((len(y), 1)), y, offset=z)[0]   # calibration-in-the-large
    brier = brier_score_loss(y, p)
    return {"mean_pred": p.mean(), "observed": prev, "citl_intercept": citl, "calib_slope": slope,
            "ECE": ece_score(y, p), "Brier": brier, "Brier_ref": prev * (1 - prev),
            "scaled_Brier": 1 - brier / (prev * (1 - prev))}

def full_metrics(y, p):
    y = np.asarray(y).astype(int); p = np.asarray(p, float)
    d = {"N": len(y), "deaths": int(y.sum()),
         "AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p),
         "Sens@90Spec": sens_at_spec(y, p)}
    d.update(calib_stats(y, p))
    return d

# ---- DeLong (fast, Sun & Xu 2014) ----
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    out = np.empty(N); out[J] = T + 1
    return out

def delong_paired(y, p1, p2):
    y = np.asarray(y).astype(int)
    order = np.argsort(-y, kind="mergesort")      # positives first
    m = int(y.sum()); n = len(y) - m
    P = np.vstack([np.asarray(p1, float), np.asarray(p2, float)])[:, order]
    tx = np.vstack([_midrank(r[:m]) for r in P]); ty = np.vstack([_midrank(r[m:]) for r in P])
    tz = np.vstack([_midrank(r) for r in P])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    l = np.array([1, -1]); var = float(l @ cov @ l)
    zstat = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else np.nan
    pval = 2 * stats.norm.sf(abs(zstat)) if var > 0 else np.nan
    return float(aucs[0]), float(aucs[1]), float(zstat), float(pval)

# ---- Bootstrap (optionally clustered by subject_id) ----
def _boot_indices(n, clusters, rng):
    if clusters is None:
        return rng.integers(0, n, n)
    uniq = np.unique(clusters)
    pick = rng.choice(uniq, size=len(uniq), replace=True)
    pos = {c: np.flatnonzero(clusters == c) for c in uniq}
    return np.concatenate([pos[c] for c in pick])

def boot_ci(y, p, fn=roc_auc_score, n_boot=N_BOOT, clusters=None, seed=SEED):
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); vals = []
    for _ in range(n_boot):
        idx = _boot_indices(len(y), clusters, rng)
        if y[idx].min() == y[idx].max():
            continue
        vals.append(fn(y[idx], p[idx]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_diff(y, p1, p2, fn=roc_auc_score, n_boot=N_BOOT, clusters=None, seed=SEED):
    rng = np.random.default_rng(seed); y = np.asarray(y); d = []
    for _ in range(n_boot):
        idx = _boot_indices(len(y), clusters, rng)
        if y[idx].min() == y[idx].max():
            continue
        d.append(fn(y[idx], p1[idx]) - fn(y[idx], p2[idx]))
    d = np.asarray(d)
    p_two = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float(min(p_two, 1.0))

# ---- Recalibration ----
def prior_shift(p, target_prev, train_prev=0.5):
    # analytic correction: model trained with a class-balanced sampler (effective prior 0.5)
    return sigmoid(logit(p) + np.log(target_prev / (1 - target_prev)) - np.log(train_prev / (1 - train_prev)))

class Platt:
    def fit(self, p, y):
        z = logit(p); self.b = fit_logistic(np.c_[np.ones_like(z), z], y); return self
    def predict(self, p):
        z = logit(p); return sigmoid(self.b[0] + self.b[1] * z)

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(np.asarray(p), np.asarray(y)); return self
    def predict(self, p):
        return np.clip(self.m.predict(np.asarray(p)), 1e-6, 1 - 1e-6)

class InterceptOnly:
    def fit(self, p, y):
        self.a = fit_logistic(np.ones((len(y), 1)), y, offset=logit(p))[0]; return self
    def predict(self, p):
        return sigmoid(logit(p) + self.a)

def crossfit_recal(y, p, maker, n_splits=5, n_repeats=20, seed=SEED):
    # out-of-fold recalibration on the evaluation set itself (supplementary; no information leak per fold)
    y = np.asarray(y).astype(int); p = np.asarray(p, float); rows = []; oof0 = None
    for r in range(n_repeats):
        oof = np.zeros_like(p)
        for tr, te in StratifiedKFold(n_splits, shuffle=True, random_state=seed + r).split(p, y):
            oof[te] = maker().fit(p[tr], y[tr]).predict(p[te])
        rows.append(calib_stats(y, oof))
        if oof0 is None:
            oof0 = oof
    return pd.DataFrame(rows).mean().to_dict(), oof0

def reliability_plot(curves, y, title, fname, n_bins=10):
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    for name, p in curves.items():
        p = np.asarray(p); bins = np.linspace(0, 1, n_bins + 1)
        idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
        xs, ys = [], []
        for b in range(n_bins):
            m = idx == b
            if m.sum() >= 5:
                xs.append(p[m].mean()); ys.append(np.asarray(y)[m].mean())
        ax.plot(xs, ys, "o-", ms=4, label=f"{name} (Brier={brier_score_loss(y, p):.3f})")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed mortality")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_title(title, fontsize=10); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(OUT / fname, dpi=300); plt.show(); plt.close(fig)

print("Utilities ready")

In [ ]:
# CELL 3 — LOAD CACHE, IDENTIFIERS, SAVED PREDICTIONS + ALIGNMENT CHECKS
cache = np.load(CACHE_FILE, allow_pickle=True)
X_temporal_raw = cache["X_temporal"].astype(np.float32)
X_static_raw   = cache["X_static"].astype(np.float32)
labels_all     = cache["labels"].astype(int)
years_all      = cache["years"].astype(int)
N_ALL = len(labels_all)
print("Cache:", X_temporal_raw.shape, X_static_raw.shape, "deaths", labels_all.sum())

HAS_IDS = ID_FILE.exists()
if HAS_IDS:
    ids = pd.read_csv(ID_FILE)
    HAS_IDS = (len(ids) == N_ALL and np.array_equal(ids["label"].to_numpy().astype(int), labels_all)
               and np.array_equal(ids["real_admit_year"].to_numpy().astype(int), years_all))
    print("Identifier file aligned with cache order:", "OK" if HAS_IDS else "NO — subject-level analyses will be skipped")
if not HAS_IDS:
    print("No aligned identifier file: building placeholder IDs (A0/A1 subject & 48h analyses skipped).")
    ids = pd.DataFrame({"subject_id": np.arange(N_ALL), "hadm_id": np.arange(N_ALL), "stay_id": np.arange(N_ALL),
                        "real_admit_year": years_all, "label": labels_all})
    RUN_COHORT_AUDIT = False

split_of = np.where(np.isin(years_all, list(TRAIN_YEARS)), "train",
           np.where(np.isin(years_all, list(VAL_YEARS)), "val",
           np.where(np.isin(years_all, list(TEST_YEARS)), "test", "none")))
ids["split"] = split_of
test_mask, val_mask, train_mask = split_of == "test", split_of == "val", split_of == "train"
ids_test = ids[test_mask].reset_index(drop=True)
y_test = ids_test["label"].to_numpy().astype(int)
print({s: int((split_of == s).sum()) for s in ["train", "val", "test"]})

PREDS = {}
for name, (fn, key) in PRED_FILES.items():
    f = PRED_DIR / fn
    if not f.exists():
        print("missing:", fn); continue
    d = np.load(f, allow_pickle=True)
    assert np.array_equal(d["test_labels"].astype(int), y_test), f"{fn}: test label/order mismatch"
    PREDS[name] = d[key].astype(np.float64)

GATING = {}   # (arm, seed) -> dict(single, ensemble)
for arm in ["ON", "OFF"]:
    for s in GATING_SEEDS:
        f = REV_DIR / f"gating_{arm}_seed{s}_predictions.npz"
        if f.exists():
            d = np.load(f, allow_pickle=True)
            assert np.array_equal(d["test_labels"].astype(int), y_test)
            GATING[(arm, s)] = {"single": d["probs_single"].astype(float), "ensemble": d["probs_ensemble"].astype(float)}
print("Loaded predictions:", list(PREDS), "| gating artefacts:", sorted(GATING))
show(pd.DataFrame([{"Model": k, "AUROC": roc_auc_score(y_test, v), "AUPRC": average_precision_score(y_test, v),
                    "Brier": brier_score_loss(y_test, v), "min p": v.min(), "max p": v.max(), "mean p": v.mean()}
                   for k, v in PREDS.items()]).round(4))

In [ ]:
# CELL 8 — A4: EXACT PREPROCESSING V2 (vectorised, verified) + DATA AUDIT
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

def add_missingness_features_loop(X):          # original implementation (used only for verification)
    n, seq, feats = X.shape
    mask = (~np.isnan(X)).astype(np.float32)
    delta = np.zeros_like(X, dtype=np.float32)
    for i in range(n):
        for k in range(feats):
            last_obs = 0
            for t in range(seq):
                if not np.isnan(X[i, t, k]):
                    last_obs = t; delta[i, t, k] = 0.0
                else:
                    delta[i, t, k] = float(t - last_obs)
    delta = delta / (seq + 1e-8)
    values = X.copy()
    for i in range(n):
        for k in range(len(VITAL_SIGNS)):
            last = 0.0
            for t in range(seq):
                if not np.isnan(values[i, t, k]):
                    last = values[i, t, k]
                else:
                    values[i, t, k] = last
    values = np.nan_to_num(values, nan=0.0)
    return np.concatenate([values, mask, delta], axis=-1).astype(np.float32)

def add_missingness_features(X):               # vectorised, identical output
    n, seq, feats = X.shape
    obs = ~np.isnan(X)
    mask = obs.astype(np.float32)
    t = np.arange(seq, dtype=np.int64)[None, :, None]
    last_obs = np.maximum.accumulate(np.where(obs, t, 0), axis=1)
    delta = np.where(obs, 0.0, (t - last_obs)).astype(np.float32) / np.float32(seq + 1e-8)
    values = X.copy()
    nv = len(VITAL_SIGNS)
    v = values[:, :, :nv]; ov = obs[:, :, :nv]
    idx = np.maximum.accumulate(np.where(ov, t[:, :, :1].repeat(nv, axis=2), -1), axis=1)
    filled = np.take_along_axis(np.nan_to_num(v, nan=0.0), np.clip(idx, 0, None), axis=1)
    values[:, :, :nv] = np.where(idx >= 0, filled, 0.0)
    values = np.nan_to_num(values, nan=0.0)
    return np.concatenate([values, mask, delta], axis=-1).astype(np.float32)

if RUN_CHECKPOINT_INFERENCE:
    # ---- verify vectorised == loop on a subset (after the same lab handling as training) ----
    sub = X_temporal_raw[:300].copy()
    sub[:, :, len(VITAL_SIGNS):] = np.nan_to_num(sub[:, :, len(VITAL_SIGNS):], nan=0.0, posinf=0.0, neginf=0.0)
    assert np.allclose(add_missingness_features(sub), add_missingness_features_loop(sub), atol=1e-7)
    print("Vectorised missingness features identical to original loop: OK")

    def preprocess_v2(X_temporal, X_static, labels, years):
        n_vital = len(VITAL_SIGNS); splits_raw = {}
        for name, yrs in [("train", TRAIN_YEARS), ("val", VAL_YEARS), ("test", TEST_YEARS)]:
            m = np.isin(years, list(yrs))
            splits_raw[name] = {"X_temp": X_temporal[m].copy(), "X_stat": X_static[m].copy(), "y": labels[m].copy()}
        audit = {"lab_nan_fraction_raw_train": float(np.isnan(splits_raw["train"]["X_temp"][:, :, n_vital:]).mean())}
        imp = IterativeImputer(max_iter=5, random_state=SEED, n_nearest_features=10)
        tr = splits_raw["train"]["X_temp"]; n_tr, seq, n_feat = tr.shape
        lab_train = tr[:, :, n_vital:].reshape(n_tr * seq, n_feat - n_vital)
        lab_train = np.nan_to_num(lab_train, nan=0.0, posinf=0.0, neginf=0.0)   # <- as in original code (before MICE)
        audit["lab_nan_fraction_entering_MICE"] = float(np.isnan(lab_train).mean())
        imp.fit(lab_train)
        for name in splits_raw:
            arr = splits_raw[name]["X_temp"]; n_sp = arr.shape[0]
            lab_d = np.nan_to_num(arr[:, :, n_vital:].reshape(n_sp * seq, n_feat - n_vital), nan=0.0, posinf=0.0, neginf=0.0)
            out = imp.transform(lab_d)
            if name == "train":
                audit["MICE_max_abs_change_train"] = float(np.abs(out - lab_d).max())
            arr[:, :, n_vital:] = out.reshape(n_sp, seq, n_feat - n_vital)
        splits = {}
        for name in splits_raw:
            splits[name] = {"X_temp": add_missingness_features(splits_raw[name]["X_temp"]),
                            "X_stat": splits_raw[name]["X_stat"].copy(), "y": splits_raw[name]["y"].copy()}
        tr_flat = splits["train"]["X_temp"][:, :, :N_FEATURES].reshape(-1, N_FEATURES)
        feat_mean = tr_flat.mean(axis=0); feat_std = tr_flat.std(axis=0) + 1e-8
        stat_mean = splits["train"]["X_stat"].mean(axis=0); stat_std = splits["train"]["X_stat"].std(axis=0) + 1e-8
        for name in splits:
            arr = splits[name]["X_temp"].copy()
            arr[:, :, :N_FEATURES] = (arr[:, :, :N_FEATURES] - feat_mean) / feat_std
            splits[name]["X_temp"] = arr
            splits[name]["X_stat"] = (splits[name]["X_stat"] - stat_mean) / stat_std
        norm = {"feat_mean": feat_mean, "feat_std": feat_std, "stat_mean": stat_mean, "stat_std": stat_std}
        return splits, norm, audit

    t0 = time.time()
    SPLITS, NORM, PREP_AUDIT = preprocess_v2(X_temporal_raw.copy(), X_static_raw.copy(), labels_all, years_all)
    print(f"preprocess_v2 rebuilt in {(time.time()-t0)/60:.1f} min")
    assert np.array_equal(SPLITS["test"]["y"].astype(int), y_test)

    tr = SPLITS["train"]["X_temp"]; nv = len(VITAL_SIGNS)
    lab_mask = tr[:, :, N_FEATURES + nv: 2 * N_FEATURES]; lab_delta = tr[:, :, 2 * N_FEATURES + nv:]
    vit_mask = tr[:, :, N_FEATURES: N_FEATURES + nv]
    temp = X_temporal_raw[:, :, VITAL_SIGNS.index("temperature")]; temp = temp[~np.isnan(temp)]
    PREP_AUDIT.update({
        "lab_mask_unique_values": np.unique(lab_mask).tolist(),
        "lab_delta_all_zero": bool(np.all(lab_delta == 0)),
        "vital_mask_observed_fraction": float(vit_mask.mean()),
        "temperature_values_below_45 (Celsius-like) %": float(100 * (temp < 45).mean()),
        "temperature_values_above_80 (Fahrenheit-like) %": float(100 * (temp > 80).mean()),
    })
    print(json.dumps(PREP_AUDIT, indent=2))
    pd.Series(PREP_AUDIT, dtype=object).to_csv(OUT / "A4_preprocessing_audit.csv")
    if PREP_AUDIT["lab_nan_fraction_entering_MICE"] == 0.0 and PREP_AUDIT["lab_delta_all_zero"]:
        FLAGS.append(
            "[CHECK] In the preprocess_v2 version reproduced here (validated against the checkpoint in A5), lab NaNs are zero-filled "
            f"before IterativeImputer (max MICE change {PREP_AUDIT['MICE_max_abs_change_train']:.2e}); lab masks are constant 1 and lab "
            "delta-t constant 0, so informative missingness/delta-t comes from the 7 vital signs. Verify against Sec. III-D/E, "
            "Table VIII and Table XII.")
    if PREP_AUDIT["temperature_values_below_45 (Celsius-like) %"] > 1 and PREP_AUDIT["temperature_values_above_80 (Fahrenheit-like) %"] > 1:
        FLAGS.append("[CHECK] Temperature channel mixes °F (itemid 223761) and °C (223762) values; mention as a limitation or harmonise in future work.")
    for f in FLAGS[-2:]: print(f)

## Experiment 2 — eICU external domain-shift audit (frozen checkpoint, no retraining)

Rebuild exactly the **corrected eICU pipeline** from the approved final notebook. The raw eICU `X_raw` and MIMIC `X_temporal_raw` will be compared **before scaling or imputation** where feasible to avoid hiding missingness. Comparisons are descriptive: missingness fractions, feature means/medians and quantiles, and standardized mean difference at **patient level** (patient 48-hour feature means). Features absent in one domain can have imprecise comparisons. No causal attribution is made.

In [ ]:
# CELL 1 — SETUP + FROZEN CONTRACT

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import math, gc, re, gzip
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")
EICU_DIR = ROOT / "mimic-iv-2.1/revision2_outputs/external_eicu/eicu_full"
CKPT_PATH = ROOT / "mimic-iv-2.1/saved_models/dsrnn_v2_best.pt"

VITAL_SIGNS = ['heart_rate','sbp','dbp','map','resp_rate','spo2','temperature']
LAB_FEATURES = [
    'creatinine','bnp','lactate','troponin','sodium','potassium','chloride',
    'bicarbonate','bun','glucose','hemoglobin','wbc','platelets','inr','pt',
    'ptt','ph','pao2','paco2','fio2','albumin','bilirubin_total','alt','ast'
]
TEMPORAL_FEATURES = VITAL_SIGNS + LAB_FEATURES
STATIC_FEATURES = [
    'age','gender_male','admission_emergency','admission_urgent',
    'elixhauser_score','hypertension','diabetes','ckd','copd','af','prior_hf'
]

assert len(TEMPORAL_FEATURES) == 31
assert len(STATIC_FEATURES) == 11
assert CKPT_PATH.exists()
assert EICU_DIR.exists()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
norm = ckpt["norm_stats"]

feat_mean = np.asarray(norm["feat_mean"], dtype=np.float32)
feat_std  = np.asarray(norm["feat_std"], dtype=np.float32)
stat_mean = np.asarray(norm["stat_mean"], dtype=np.float32)
stat_std  = np.asarray(norm["stat_std"], dtype=np.float32)

print("Device:", DEVICE)
print("eICU:", EICU_DIR)
print("Checkpoint:", CKPT_PATH)
print("Frozen internal AUROC:", ckpt.get("test_auc_ensemble"))
print("Training temperature mean:", feat_mean[TEMPORAL_FEATURES.index("temperature")])
print("Training FiO2 mean:", feat_mean[TEMPORAL_FEATURES.index("fio2")])


In [ ]:
# CELL 2 — LOAD CORE eICU TABLES

def read_gz(name, usecols=None):
    return pd.read_csv(
        EICU_DIR / name, compression="gzip",
        low_memory=False, usecols=usecols
    )

patient = read_gz("patient.csv.gz")
diagnosis = read_gz("diagnosis.csv.gz")
admissiondx = read_gz("admissionDx.csv.gz")
pasthistory = read_gz("pastHistory.csv.gz")

print("patient     :", patient.shape)
print("diagnosis   :", diagnosis.shape)
print("admissionDx :", admissiondx.shape)
print("pastHistory :", pasthistory.shape)


In [ ]:
# ============================================================
# CELL 3 — HEART-FAILURE COHORT + OUTCOME
# FIXED: pandas pd.NA-safe mortality construction
# UNKNOWN HOSPITAL OUTCOME PRESERVED AS NaN
# ============================================================

dx_text = diagnosis["diagnosisstring"].fillna("").astype(str)

adx_cols = [
    c for c in ["admitdxpath", "admitdxname", "admitdxtext"]
    if c in admissiondx.columns
]

adx_text = (
    admissiondx[adx_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

hf_regex = (
    r"heart failure|congestive heart failure|"
    r"\bchf\b|cardiomyopathy.*heart failure"
)

# ------------------------------------------------------------
# 1. Identify HF ICU stays
# ------------------------------------------------------------

hf_ids = set(
    diagnosis.loc[
        dx_text.str.contains(
            hf_regex,
            case=False,
            regex=True,
            na=False
        ),
        "patientunitstayid"
    ]
    .dropna()
    .astype(int)
)

hf_ids |= set(
    admissiondx.loc[
        adx_text.str.contains(
            hf_regex,
            case=False,
            regex=True,
            na=False
        ),
        "patientunitstayid"
    ]
    .dropna()
    .astype(int)
)

print(f"HF candidate ICU stays: {len(hf_ids):,}")


# ------------------------------------------------------------
# 2. Build cohort
# ------------------------------------------------------------

cohort = patient[
    patient["patientunitstayid"].isin(hf_ids)
].copy()


# ------------------------------------------------------------
# 3. Adult patients
# ------------------------------------------------------------

def parse_age(x):

    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s in ["> 89", ">89"]:
        return 90.0

    return pd.to_numeric(
        s,
        errors="coerce"
    )


cohort["age_num"] = cohort["age"].map(parse_age)

cohort = cohort[
    cohort["age_num"] >= 18
].copy()


# ------------------------------------------------------------
# 4. ICU stay >24 h
# ------------------------------------------------------------

cohort["unitdischargeoffset_num"] = pd.to_numeric(
    cohort["unitdischargeoffset"],
    errors="coerce"
)

cohort = cohort[
    cohort["unitdischargeoffset_num"] > 24 * 60
].copy()


# ------------------------------------------------------------
# 5. One ICU stay per hospital stay
# ------------------------------------------------------------

if "patienthealthsystemstayid" in cohort.columns:

    cohort["unitvisitnumber_num"] = pd.to_numeric(
        cohort["unitvisitnumber"],
        errors="coerce"
    )

    cohort = (
        cohort
        .sort_values(
            [
                "patienthealthsystemstayid",
                "unitvisitnumber_num",
                "patientunitstayid"
            ]
        )
        .drop_duplicates(
            "patienthealthsystemstayid",
            keep="first"
        )
        .copy()
    )

else:

    cohort = (
        cohort
        .sort_values("patientunitstayid")
        .drop_duplicates(
            "uniquepid",
            keep="first"
        )
        .copy()
    )


# ------------------------------------------------------------
# 6. Hospital mortality endpoint
#
# IMPORTANT:
# Alive   -> 0
# Expired -> 1
# Missing / unknown -> NaN
#
# Do NOT use np.where() directly with pandas nullable Boolean.
# ------------------------------------------------------------

status = (
    cohort["hospitaldischargestatus"]
    .astype("string")
    .str.lower()
    .str.strip()
)

# Start ALL outcomes as unknown
cohort["mortality"] = np.nan

# Assign only known outcomes
cohort.loc[
    status.fillna("").eq("alive"),
    "mortality"
] = 0.0

cohort.loc[
    status.fillna("").eq("expired"),
    "mortality"
] = 1.0


# ------------------------------------------------------------
# 7. AUDIT
# ------------------------------------------------------------

n_total = len(cohort)
n_known = int(cohort["mortality"].notna().sum())
n_unknown = int(cohort["mortality"].isna().sum())
n_deaths = int((cohort["mortality"] == 1).sum())
n_alive = int((cohort["mortality"] == 0).sum())

mortality_rate = (
    cohort["mortality"].mean() * 100
    if n_known > 0
    else np.nan
)

print("\n" + "=" * 90)
print("eICU HEART-FAILURE COHORT")
print("=" * 90)

print(f"HF cohort                : {n_total:,}")
print(f"Known hospital outcomes  : {n_known:,}")
print(f"Unknown hospital outcomes: {n_unknown:,}")
print(f"Alive                     : {n_alive:,}")
print(f"Deaths                    : {n_deaths:,}")
print(f"Known-outcome mortality   : {mortality_rate:.2f}%")

print("\nHospital discharge status:")
print(
    status
    .fillna("<MISSING>")
    .value_counts(dropna=False)
)

print("\n" + "=" * 90)
print("✓ HF COHORT CONSTRUCTION COMPLETE")
print("✓ UNKNOWN OUTCOMES PRESERVED AS NaN")
print("✓ NO OUTCOME IMPUTATION")
print("=" * 90)

In [ ]:
# CELL 4 — VITALS 0–48 h + UNIT HARMONIZATION

ids = set(cohort["patientunitstayid"].astype(int))

vp = read_gz("vitalPeriodic.csv.gz")
vp = vp[vp["patientunitstayid"].isin(ids)].copy()
vp["observationoffset"] = pd.to_numeric(vp["observationoffset"], errors="coerce")
vp = vp[(vp["observationoffset"] >= 0) & (vp["observationoffset"] < 48*60)].copy()
vp["hour"] = (vp["observationoffset"] // 60).astype(int)

vmap = {
    "heart_rate":"heartrate",
    "sbp":"systemicsystolic",
    "dbp":"systemicdiastolic",
    "map":"systemicmean",
    "resp_rate":"respiration",
    "spo2":"sao2",
    "temperature":"temperature"
}
for dst, src in vmap.items():
    vp[dst] = pd.to_numeric(vp[src], errors="coerce") if src in vp.columns else np.nan

# Physiologic cleanup, before aggregation
ranges = {
    "heart_rate": (20, 250),
    "sbp": (40, 300),
    "dbp": (20, 200),
    "map": (30, 220),
    "resp_rate": (4, 80),
    "spo2": (50, 100),
}
for f, (lo, hi) in ranges.items():
    vp.loc[~vp[f].between(lo, hi), f] = np.nan

# eICU periodic temperature is commonly Celsius-like in this extract.
# The frozen training mean (~87.8) is Fahrenheit-space.
temp_med = vp["temperature"].median(skipna=True)
print("Raw eICU temperature median:", temp_med)

if pd.notna(temp_med) and temp_med < 60:
    vp["temperature"] = vp["temperature"] * 9.0/5.0 + 32.0
    print("✓ Temperature converted C -> F")
else:
    print("✓ Temperature already appears Fahrenheit-like")

vp.loc[~vp["temperature"].between(77, 113), "temperature"] = np.nan

vital_hour = (
    vp.groupby(["patientunitstayid","hour"])[VITAL_SIGNS]
      .median()
      .reset_index()
)

print("Vital hourly rows:", len(vital_hour))
for f in VITAL_SIGNS:
    print(f"{f:15s}: patients={vital_hour.loc[vital_hour[f].notna(),'patientunitstayid'].nunique():,}")


In [ ]:
# CELL 5 — LABS 0–48 h + VERIFIED NAME MAPPING + UNIT HARMONIZATION

usecols = ["patientunitstayid","labresultoffset","labname","labresult"]
lab = pd.read_csv(
    EICU_DIR / "lab.csv.gz",
    compression="gzip", usecols=usecols, low_memory=False
)

lab["patientunitstayid"] = pd.to_numeric(lab["patientunitstayid"], errors="coerce")
lab["labresultoffset"] = pd.to_numeric(lab["labresultoffset"], errors="coerce")
lab["labresult"] = pd.to_numeric(lab["labresult"], errors="coerce")

lab = lab[
    lab["patientunitstayid"].isin(ids) &
    (lab["labresultoffset"] >= 0) &
    (lab["labresultoffset"] < 48*60)
].copy()

lab["hour"] = (lab["labresultoffset"] // 60).astype(int)
lab["labname_clean"] = lab["labname"].astype(str).str.lower().str.strip()

LAB_ALIASES = {
    "creatinine":["creatinine"],
    "bnp":["bnp","brain natriuretic peptide"],
    "lactate":["lactate","lactic acid"],
    "troponin":["troponin - i","troponin i","troponin-i","troponin - t","troponin t","troponin-t"],
    "sodium":["sodium"],
    "potassium":["potassium"],
    "chloride":["chloride"],
    "bicarbonate":["bicarbonate","hco3"],
    "bun":["bun","blood urea nitrogen"],
    "glucose":["glucose"],
    "hemoglobin":["hgb","hemoglobin"],
    "wbc":["wbc x 1000","wbc","white blood cell count"],
    "platelets":["platelets x 1000","platelets","platelet count"],
    "inr":["pt - inr"],
    "pt":["pt"],
    "ptt":["ptt","aptt"],
    "ph":["ph"],
    "pao2":["pao2"],
    "paco2":["paco2"],
    "fio2":["fio2"],
    "albumin":["albumin"],
    "bilirubin_total":["total bilirubin","bilirubin total","bilirubin, total"],
    "alt":["alt (sgpt)","alt","sgpt"],
    "ast":["ast (sgot)","ast","sgot"]
}
alias_to_feature = {
    a.lower().strip(): f
    for f, aliases in LAB_ALIASES.items()
    for a in aliases
}
lab["feature"] = lab["labname_clean"].map(alias_to_feature)
mapped = lab[lab["feature"].notna() & lab["labresult"].notna()].copy()

# FiO2 -> fraction 0–1.
fio = mapped["feature"].eq("fio2")
mapped.loc[fio & mapped["labresult"].between(1.5, 100), "labresult"] /= 100.0
mapped.loc[fio & ~mapped["labresult"].between(0.15, 1.0), "labresult"] = np.nan

lab_long = (
    mapped.dropna(subset=["labresult"])
          .groupby(["patientunitstayid","hour","feature"], as_index=False)["labresult"]
          .median()
)

lab_hour = (
    lab_long.pivot(index=["patientunitstayid","hour"], columns="feature", values="labresult")
            .reset_index()
)
lab_hour.columns.name = None

for f in LAB_FEATURES:
    if f not in lab_hour.columns:
        lab_hour[f] = np.nan
lab_hour = lab_hour[["patientunitstayid","hour"] + LAB_FEATURES]

print("Lab hourly rows:", len(lab_hour))
print("Mapped lab patients:", lab_hour["patientunitstayid"].nunique())
print("FiO2 observed range:",
      lab_hour["fio2"].min(skipna=True), "to", lab_hour["fio2"].max(skipna=True))

del lab, mapped, lab_long
gc.collect()


In [ ]:
# ============================================================
# CELL 6 — STATIC FEATURES — FIXED
# COST Improved eICU External Validation
#
# IMPORTANT:
# Prior comorbidity flags use pastHistory ONLY.
# Current-stay diagnosis is NOT used to define prior history.
# ============================================================

print("=" * 90)
print("BUILDING eICU STATIC FEATURES")
print("=" * 90)

# ------------------------------------------------------------
# 1. Start from HF cohort
# ------------------------------------------------------------

static = (
    cohort[["patientunitstayid"]]
    .drop_duplicates()
    .copy()
)

print(f"Cohort patients: {len(static):,}")


# ============================================================
# 2. DEMOGRAPHICS + ADMISSION FEATURES
# ============================================================

p = patient[
    [
        "patientunitstayid",
        "age",
        "gender",
        "hospitaladmitsource",
        "unitadmitsource"
    ]
].copy()

p["age"] = p["age"].map(parse_age)


# ------------------------------------------------------------
# Gender
# FIX: avoid np.where() with pandas nullable Boolean
# ------------------------------------------------------------

gender_clean = (
    p["gender"]
    .astype("string")
    .str.lower()
    .str.strip()
)

p["gender_male"] = np.nan

p.loc[
    gender_clean.fillna("").eq("male"),
    "gender_male"
] = 1.0

p.loc[
    gender_clean.fillna("").eq("female"),
    "gender_male"
] = 0.0


# ------------------------------------------------------------
# Admission emergency
#
# Frozen COST training statistics show this feature was
# constant zero. Preserve training-space state.
# ------------------------------------------------------------

p["admission_emergency"] = 0.0


# ------------------------------------------------------------
# Admission urgent
# ------------------------------------------------------------

src = (
    p["hospitaladmitsource"]
    .fillna("")
    .astype(str)
    + " " +
    p["unitadmitsource"]
    .fillna("")
    .astype(str)
).str.lower()

p["admission_urgent"] = (
    src
    .str.contains(
        r"\burgent\b",
        regex=True,
        na=False
    )
    .astype(float)
)


# ------------------------------------------------------------
# Merge demographic/admission variables
# ------------------------------------------------------------

static = static.merge(
    p[
        [
            "patientunitstayid",
            "age",
            "gender_male",
            "admission_emergency",
            "admission_urgent"
        ]
    ],
    on="patientunitstayid",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 3. PRIOR MEDICAL HISTORY
# pastHistory ONLY
# ============================================================

ph = pasthistory[
    [
        "patientunitstayid",
        "pasthistorypath",
        "pasthistoryvalue",
        "pasthistoryvaluetext"
    ]
].copy()

ph["text"] = (
    ph["pasthistorypath"]
    .fillna("")
    .astype(str)
    + " " +
    ph["pasthistoryvalue"]
    .fillna("")
    .astype(str)
    + " " +
    ph["pasthistoryvaluetext"]
    .fillna("")
    .astype(str)
).str.lower()


# Combine history for each ICU stay
ph_text = (
    ph.groupby(
        "patientunitstayid"
    )["text"]
    .apply(" ".join)
)


# ============================================================
# 4. COMORBIDITY DEFINITIONS
# ============================================================

patterns = {

    "hypertension":
        r"hypertension|hypertensive",

    "diabetes":
        r"diabetes|diabetic",

    "ckd":
        r"chronic kidney|chronic renal|renal failure",

    "copd":
        r"copd|chronic obstructive|emphysema",

    "af":
        r"atrial fibrillation|atrial fib",

    "prior_hf":
        r"heart failure|congestive heart failure|\bchf\b"
}


comorb = pd.DataFrame(
    index=ph_text.index
)

for f, pat in patterns.items():

    comorb[f] = (
        ph_text
        .str.contains(
            pat,
            regex=True,
            na=False
        )
        .astype(float)
    )

comorb = comorb.reset_index()


# ============================================================
# 5. MERGE COMORBIDITIES
# ============================================================

static = static.merge(
    comorb,
    on="patientunitstayid",
    how="left"
)

for f in patterns.keys():

    static[f] = (
        static[f]
        .fillna(0.0)
        .astype(np.float32)
    )


# ============================================================
# 6. COST-COMPATIBLE ELIXHAUSER PROXY
# ============================================================

static["elixhauser_score"] = (
    static[
        list(patterns.keys())
    ]
    .sum(axis=1)
    .astype(np.float32)
)


# ============================================================
# 7. FINAL EXACT FEATURE ORDER
# ============================================================

static = static[
    ["patientunitstayid"] + STATIC_FEATURES
].copy()


# ============================================================
# 8. AUDIT
# ============================================================

print("\n" + "=" * 90)
print("STATIC FEATURE AUDIT")
print("=" * 90)

print(f"Patients: {len(static):,}")

for f in STATIC_FEATURES:

    missing = int(
        static[f].isna().sum()
    )

    if f == "age":

        print(
            f"{f:22s}: "
            f"mean={static[f].mean():8.3f} | "
            f"missing={missing:,}"
        )

    else:

        print(
            f"{f:22s}: "
            f"mean={static[f].mean():8.4f} | "
            f"missing={missing:,}"
        )


print("\nGender:")
print(
    gender_clean
    .fillna("<MISSING>")
    .value_counts(dropna=False)
)


print("\nElixhauser proxy:")
print(
    static["elixhauser_score"]
    .describe()
)


print("\n" + "=" * 90)
print("✓ STATIC FEATURE TABLE COMPLETE")
print("✓ PRIOR COMORBIDITIES FROM pastHistory ONLY")
print("✓ CURRENT-STAY DIAGNOSIS NOT USED FOR PRIOR HISTORY")
print("✓ NO RETRAINING")
print("=" * 90)

In [ ]:
# CELL 7 — 48×31 GRID + MINIMUM-DATA FILTER + KNOWN OUTCOME FILTER

all_ids = np.sort(cohort["patientunitstayid"].astype(int).unique())
grid = pd.MultiIndex.from_product(
    [all_ids, range(48)],
    names=["patientunitstayid","hour"]
).to_frame(index=False)

temporal = (
    grid.merge(vital_hour, on=["patientunitstayid","hour"], how="left")
        .merge(lab_hour, on=["patientunitstayid","hour"], how="left")
)

vital_counts = temporal.groupby("patientunitstayid")[VITAL_SIGNS].count().sum(axis=1)
lab_counts = temporal.groupby("patientunitstayid")[LAB_FEATURES].count().sum(axis=1)

eligible = (vital_counts >= 6) & (lab_counts >= 2)
eligible_ids = eligible[eligible].index.astype(int)

cohort_e = cohort[cohort["patientunitstayid"].isin(eligible_ids)].copy()
cohort_e = cohort_e[cohort_e["mortality"].notna()].copy()

final_ids = np.sort(cohort_e["patientunitstayid"].astype(int).unique())

temporal_e = temporal[temporal["patientunitstayid"].isin(final_ids)].copy()
static_e = static[static["patientunitstayid"].isin(final_ids)].copy()

cohort_e = cohort_e.set_index("patientunitstayid").loc[final_ids].reset_index()
static_e = static_e.set_index("patientunitstayid").loc[final_ids].reset_index()
temporal_e = temporal_e.sort_values(["patientunitstayid","hour"]).reset_index(drop=True)

assert len(temporal_e) == len(final_ids)*48
assert len(static_e) == len(final_ids)

y_ext = cohort_e["mortality"].astype(np.float32).to_numpy()

print("Final known-outcome eligible cohort:", len(final_ids))
print("Deaths:", int(y_ext.sum()))
print("Mortality:", f"{y_ext.mean()*100:.2f}%")


In [ ]:
# CELL 8 — EXACT TRAINING-ORDER PREPROCESSING CORRECTION
# Reproduces the recovered original preprocess_v2 ordering.

N = len(final_ids)

X_raw = (
    temporal_e[TEMPORAL_FEATURES]
    .to_numpy(np.float32)
    .reshape(N, 48, 31)
)
S_raw = static_e[STATIC_FEATURES].to_numpy(np.float32)

# BNP was constant zero in frozen training space.
bnp_idx = TEMPORAL_FEATURES.index("bnp")
X_raw[:,:,bnp_idx] = 0.0

# ------------------------------------------------------------------
# ORIGINAL TRAINING ORDER:
# 1) LABS: nan_to_num BEFORE IterativeImputer.
#    Since no NaNs remain, the fitted IterativeImputer is effectively
#    an identity transform for the lab matrix.
# 2) add_missingness_features() runs AFTER this operation.
# ------------------------------------------------------------------

X_pre = X_raw.copy()
n_vital = len(VITAL_SIGNS)

# Training code zero-filled ALL missing lab values before imputer.
X_pre[:,:,n_vital:] = np.nan_to_num(
    X_pre[:,:,n_vital:],
    nan=0.0, posinf=0.0, neginf=0.0
)

# Mask is computed AFTER lab zero-fill, exactly like training.
mask = (~np.isnan(X_pre)).astype(np.float32)

# Delta computed AFTER lab zero-fill.
delta = np.zeros_like(X_pre, dtype=np.float32)
for i in range(N):
    for k in range(31):
        last_obs = 0
        for t in range(48):
            if not np.isnan(X_pre[i,t,k]):
                last_obs = t
                delta[i,t,k] = 0.0
            else:
                delta[i,t,k] = float(t-last_obs)
delta /= 48.0

# Vital forward-fill; labs are already zero-filled.
values = X_pre.copy()
for i in range(N):
    for k in range(n_vital):
        last = 0.0
        for t in range(48):
            if not np.isnan(values[i,t,k]):
                last = values[i,t,k]
            else:
                values[i,t,k] = last

values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)

# Frozen MIMIC normalization
values_norm = (values - feat_mean.reshape(1,1,-1)) / feat_std.reshape(1,1,-1)
X_ext = np.concatenate([values_norm, mask, delta], axis=-1).astype(np.float32)

# Static missing values -> frozen training mean (normalized 0), NOT eICU mean.
for j in range(11):
    miss = ~np.isfinite(S_raw[:,j])
    S_raw[miss,j] = stat_mean[j]

S_ext = ((S_raw - stat_mean.reshape(1,-1)) / stat_std.reshape(1,-1)).astype(np.float32)

assert X_ext.shape == (N,48,93)
assert S_ext.shape == (N,11)
assert np.isfinite(X_ext).all()
assert np.isfinite(S_ext).all()

# Critical audit proving corrected training-order behavior
lab_mask = X_ext[:,:,31+n_vital:62]
lab_delta = X_ext[:,:,62+n_vital:93]

print("="*100)
print("CORRECTED PREPROCESSING AUDIT")
print("="*100)
print("X_ext:", X_ext.shape)
print("S_ext:", S_ext.shape)
print("Lab mask unique:", np.unique(lab_mask))
print("Lab delta unique:", np.unique(lab_delta))
print("Temperature normalized observed/filled range:",
      float(X_ext[:,:,TEMPORAL_FEATURES.index('temperature')].min()),
      "to",
      float(X_ext[:,:,TEMPORAL_FEATURES.index('temperature')].max()))
print("FiO2 normalized range:",
      float(X_ext[:,:,TEMPORAL_FEATURES.index('fio2')].min()),
      "to",
      float(X_ext[:,:,TEMPORAL_FEATURES.index('fio2')].max()))

assert np.array_equal(np.unique(lab_mask), np.array([1.], dtype=np.float32))
assert np.all(lab_delta == 0)

print("✓ Original training-order lab mask/delta behavior reproduced")
print("✓ Frozen MIMIC normalization only")
print("✓ No eICU fitting")


In [ ]:
# CELL 9 — EXACT COST IMPROVED ARCHITECTURE + STRICT LOAD

class SinusoidalPE(nn.Module):
    def __init__(self,d_model,max_len=256,dropout=0.1):
        super().__init__()
        self.dropout=nn.Dropout(dropout)
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(max_len).unsqueeze(1).float()
        div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x):
        return self.dropout(x+self.pe[:,:x.size(1)])

class StochasticDepth(nn.Module):
    def __init__(self,p=0.1):
        super().__init__(); self.p=p
    def forward(self,x,residual):
        if not self.training or self.p==0: return x+residual
        keep=torch.bernoulli(torch.tensor(1-self.p)).item()
        return x+residual*keep

class ClinicalSalienceV2(nn.Module):
    def __init__(self,input_dim,hidden_dim,n_features):
        super().__init__()
        self.fc1=nn.Linear(input_dim,hidden_dim*2)
        self.norm=nn.LayerNorm(hidden_dim)
        self.fc2=nn.Linear(hidden_dim,n_features)
        self.tau=nn.Parameter(torch.tensor(0.5))
    def forward(self,x_t,h_prev):
        h=self.fc1(torch.cat([x_t,h_prev],-1))
        h,gate=h.chunk(2,-1)
        h=self.norm(h*torch.sigmoid(gate))
        s=torch.sigmoid(self.fc2(h))
        hard=(s>self.tau).float()
        return hard-s.detach()+s,s

class TemporalMultiHeadAttn(nn.Module):
    def __init__(self,d_model,n_heads,dropout=0.1):
        super().__init__()
        self.attn=nn.MultiheadAttention(d_model,n_heads,dropout=dropout,batch_first=True)
        self.norm=nn.LayerNorm(d_model)
        self.sdrop=StochasticDepth(.1)
    def forward(self,x):
        out,w=self.attn(x,x,x)
        return self.norm(self.sdrop(x,out)),w

class StaticTemporalFusion(nn.Module):
    def __init__(self,d_temporal,n_static,d_out,n_heads=4,dropout=.1):
        super().__init__()
        self.static_proj=nn.Linear(n_static,d_temporal)
        self.cross_attn=nn.MultiheadAttention(d_temporal,n_heads,dropout=dropout,batch_first=True)
        self.norm=nn.LayerNorm(d_temporal)
        self.ffn=nn.Sequential(
            nn.Linear(d_temporal*2,d_temporal),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(d_temporal,d_out)
        )
    def forward(self,s,h):
        q=self.static_proj(s).unsqueeze(1)
        ctx,w=self.cross_attn(q,h,h)
        ctx=self.norm(ctx.squeeze(1))
        return self.ffn(torch.cat([ctx,h.mean(1)],-1)),w.squeeze(1)

class DSRNNv2(nn.Module):
    def __init__(self,n_input=93,n_static=11,hidden_size=256,n_lstm_layers=2,
                 n_heads=4,attention_dim=128,salience_hidden=128,
                 dropout=.3,lambda_sparse=5e-4):
        super().__init__()
        self.hidden_size=hidden_size
        self.lambda_sparse=lambda_sparse
        self.input_proj=nn.Sequential(
            nn.Linear(n_input,hidden_size),nn.LayerNorm(hidden_size),
            nn.GELU(),nn.Dropout(dropout)
        )
        self.pos_enc=SinusoidalPE(hidden_size,dropout=dropout*.5)
        self.salience=ClinicalSalienceV2(hidden_size*2,salience_hidden,hidden_size)
        self.lstm=nn.LSTM(
            hidden_size,hidden_size,num_layers=n_lstm_layers,batch_first=True,
            bidirectional=True,dropout=dropout
        )
        self.bidir_proj=nn.Sequential(
            nn.Linear(hidden_size*2,hidden_size),nn.LayerNorm(hidden_size),nn.GELU()
        )
        self.temporal_attn=TemporalMultiHeadAttn(hidden_size,n_heads,dropout)
        self.fusion=StaticTemporalFusion(hidden_size,n_static,attention_dim,n_heads,dropout)
        self.classifier=nn.Sequential(
            nn.LayerNorm(attention_dim),nn.Dropout(dropout),
            nn.Linear(attention_dim,attention_dim//2),nn.GELU(),
            nn.Dropout(dropout*.5),nn.Linear(attention_dim//2,1)
        )
    def forward(self,x_temporal,x_static):
        b,seq,_=x_temporal.shape
        xp=self.pos_enc(self.input_proj(x_temporal))
        hp=torch.zeros(b,self.hidden_size,device=x_temporal.device)
        gs=[]; sp=torch.tensor(0.,device=x_temporal.device)
        for t in range(seq):
            xt=xp[:,t]
            m,_=self.salience(xt,hp)
            xg=m*xt+(1-m)*xt*.1
            gs.append(xg.unsqueeze(1))
            hp=hp*.9+xg.detach()*.1
            sp+=m.norm(p=1)
        g=torch.cat(gs,1)
        sp=self.lambda_sparse*sp/(seq*b)
        lo,_=self.lstm(g)
        lo=self.bidir_proj(lo)
        H,_=self.temporal_attn(lo)
        z,_=self.fusion(x_static,H)
        return self.classifier(z).squeeze(-1),sp

cfg=ckpt.get("config",{})
model=DSRNNv2(
    n_input=93,n_static=11,
    hidden_size=cfg.get("hidden_size",256),
    n_lstm_layers=cfg.get("n_lstm_layers",2),
    n_heads=cfg.get("n_heads",4),
    attention_dim=cfg.get("attention_dim",128),
    salience_hidden=cfg.get("salience_hidden",128),
    dropout=cfg.get("dropout",.3),
    lambda_sparse=cfg.get("lambda_sparse",5e-4)
).to(DEVICE)

result=model.load_state_dict(ckpt["model_state_dict"],strict=True)
model.eval()
print(result)
print("Parameters:",sum(p.numel() for p in model.parameters()))
print("✓ strict=True; eval mode:", not model.training)


In [ ]:
# CELL 10 — FROZEN EXTERNAL INFERENCE

X = X_ext
S = S_ext
y = y_ext.astype(int)

BATCH_SIZE = 256
probs=[]

with torch.no_grad():
    for start in range(0,len(y),BATCH_SIZE):
        end=min(start+BATCH_SIZE,len(y))
        xb=torch.from_numpy(X[start:end]).to(DEVICE)
        sb=torch.from_numpy(S[start:end]).to(DEVICE)
        logits,_=model(xb,sb)
        probs.append(torch.sigmoid(logits).cpu().numpy())
        if end % 2048 < BATCH_SIZE or end == len(y):
            print(f"Processed {end:,}/{len(y):,}")

y_prob_ext=np.concatenate(probs).reshape(-1)

auroc=roc_auc_score(y,y_prob_ext)
auprc=average_precision_score(y,y_prob_ext)
brier=brier_score_loss(y,y_prob_ext)

fpr,tpr,thr=roc_curve(y,y_prob_ext)
j=np.argmax(tpr-fpr)
threshold=thr[j]
yp=(y_prob_ext>=threshold).astype(int)
tn,fp,fn,tp=confusion_matrix(y,yp).ravel()

sens=tp/(tp+fn)
spec=tn/(tn+fp)
ppv=tp/(tp+fp)
npv=tn/(tn+fn)

print("="*100)
print("CORRECTED eICU EXTERNAL VALIDATION")
print("="*100)
print(f"N                 : {len(y):,}")
print(f"Deaths            : {int(y.sum()):,}")
print(f"Mortality         : {y.mean()*100:.2f}%")
print(f"AUROC             : {auroc:.4f}")
print(f"AUPRC             : {auprc:.4f}")
print(f"Brier             : {brier:.4f}")
print(f"Youden threshold  : {threshold:.6f}")
print(f"Accuracy          : {accuracy_score(y,yp):.4f}")
print(f"Sensitivity       : {sens:.4f}")
print(f"Specificity       : {spec:.4f}")
print(f"PPV               : {ppv:.4f}")
print(f"NPV               : {npv:.4f}")
print(f"F1                : {f1_score(y,yp):.4f}")
print(f"MCC               : {matthews_corrcoef(y,yp):.4f}")
print(f"TN/FP/FN/TP       : {tn:,}/{fp:,}/{fn:,}/{tp:,}")
print("Probability range :", float(y_prob_ext.min()), "to", float(y_prob_ext.max()))
print("Internal AUROC    :", float(ckpt.get("test_auc_ensemble",np.nan)))
print("NO RETRAINING / NO FINE-TUNING / NO eICU NORMALIZATION FITTING")


In [ ]:
# DOMAIN-SHIFT REPORT — aggregate only; no export of patient rows
from scipy import stats
internal=X_temporal_raw[np.isin(years_all,list(TEST_YEARS))]
external=X_raw
assert internal.shape[1:]==external.shape[1:]==(48,31)
assert len(external)==12813, 'Not the final corrected external cohort — STOP and investigate.'
assert np.array_equal(np.asarray(y_ext).astype(int),y), 'External labels misaligned.'

def patient_feature_average(a):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',category=RuntimeWarning)
        return np.nanmean(np.where(np.isfinite(a),a,np.nan),axis=1)

m_patient=patient_feature_average(internal)
e_patient=patient_feature_average(external)
rows=[]
for j,name in enumerate(TEMPORAL_FEATURES):
    m=internal[:,:,j];e=external[:,:,j]
    x=m_patient[:,j];z=e_patient[:,j]
    x=x[np.isfinite(x)];z=z[np.isfinite(z)]
    m_missing=float((~np.isfinite(m)).mean());e_missing=float((~np.isfinite(e)).mean())
    # pooled SD across patient-averaged features, not 48h time points
    if len(x)>1 and len(z)>1:
        pooled=np.sqrt(.5*(np.var(x,ddof=1)+np.var(z,ddof=1)))
        smd=(np.mean(z)-np.mean(x))/pooled if pooled>1e-12 else np.nan
        med_m,med_e=float(np.median(x)),float(np.median(z))
    else: smd=np.nan;med_m=np.nan;med_e=np.nan
    rows.append({'feature':name,'MIMIC_observed_fraction':1-m_missing,
                 'eICU_observed_fraction':1-e_missing,'difference_eICU_minus_MIMIC':m_missing-e_missing,
                 'MIMIC_patient_median':med_m,'eICU_patient_median':med_e,
                 'standardized_mean_difference':smd,
                 'MIMIC_patients_with_data':len(x),'eICU_patients_with_data':len(z)})
shift=pd.DataFrame(rows)
shift['abs_SMD']=shift['standardized_mean_difference'].abs()
shift=shift.sort_values('abs_SMD',ascending=False,na_position='last')
shift.to_csv(TARGET_OUT/'R2_MIMIC_vs_eICU_feature_missingness_shift.csv',index=False)
print('MIMIC held-out n:',len(internal),'eICU n:',len(external))
print('External AUROC',roc_auc_score(y_ext,y_prob_ext),'AUPRC',average_precision_score(y_ext,y_prob_ext),
      'Brier',brier_score_loss(y_ext,y_prob_ext))
display(shift.head(31))


In [ ]:
# DESCRIPTIVE PERFORMANCE/CALIBRATION AUDIT (external only, no test-set fitting)
from sklearn.calibration import calibration_curve
from sklearn.metrics import log_loss

def ece_equal_width(y,p,n_bins=10):
    edges=np.linspace(0,1,n_bins+1);idx=np.minimum(np.searchsorted(edges,p,side='right')-1,n_bins-1)
    score=0.
    for b in range(n_bins):
        take=idx==b
        if take.any():score+=take.mean()*abs(y[take].mean()-p[take].mean())
    return float(score)
result={'n_external':int(len(y_ext)),'deaths_external':int(y_ext.sum()),
        'AUROC_external':float(roc_auc_score(y_ext,y_prob_ext)),
        'AUPRC_external':float(average_precision_score(y_ext,y_prob_ext)),
        'Brier_external':float(brier_score_loss(y_ext,y_prob_ext)),
        'ECE_10bins_external':ece_equal_width(y_ext.astype(int),y_prob_ext),
        'mean_predicted_risk_external':float(y_prob_ext.mean()),
        'observed_mortality_external':float(y_ext.mean())}
with open(TARGET_OUT/'R2_external_calibration_aggregate.json','w') as h:json.dump(result,h,indent=2)
print(json.dumps(result,indent=2))
print('IMPORTANT: no recalibration fitted to eICU outcomes.')


## result interpretation gate

1. **Sparse models:** Report the trained random-mask model as a distinct *independently optimized control*, separately from A6 frozen-model interventions. Its random mask **resamples every forward pass**; therefore it is a stochastic, expected-density-matched control, not a static fixed mask. The model architecture is matched, but gate-specific parameter usage is different. Do **not** claim equivalence to all sparse architectures.
2. **Domain shift:** The produced SMD and observation-rate differences quantify external heterogeneity, **not causal effects on AUROC**. Discuss any potential measurement/unit mismatch with caution (original MIMIC temperature coding has known caveats).
3. **Publication integrity:** Do not insert new table values unless all training seeds completed, test-label alignment passed, and external AUROC reproduces ≈0.6341. Report negative or null results too.
4. **What is still not tested:** A trained fixed-mask baseline; intervention-specific causal decomposition of external shift; prospective deployment.

*cannot be guaranteed* to consider the work fully answered by any particular extra experiment. The notebook improves the empirical response without manipulating the reported main model results.